# 第 14 章：模型评测

这个 notebook 对应 `lessons/14_evaluation.md`，演示离线 eval runner：定义 eval item、运行 fake predictor、保存 predictions / metrics / report / failure cases，并比较两个模型版本的指标差异。

In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory

from src.evaluation.runner import (
    EvalExample,
    compare_metric_reports,
    run_eval,
    write_regression_report,
)

## 1. Eval Set 是验收清单

每条 eval item 都要保留 expected behavior、risk tags、required citations 和格式要求。

In [ ]:
examples = [
    EvalExample(
        id="eval_contract_001",
        input="分析违约责任条款是否需要人工复核。",
        expected_behavior="输出 JSON，引用合同审查依据，并标记人工复核。",
        gold_facts=["liability cap matters"],
        required_citations=["contract_guideline#chunk_1"],
        risk_tags=["contract", "needs_human_review"],
        rubric="legal_contract_v1",
        required_json_fields=["answer", "citations", "needs_human_review"],
        source_group="contract_guideline",
    ),
    EvalExample(
        id="eval_no_answer_001",
        input="资料中没有提到量子芯片制造步骤，能否直接回答？",
        expected_behavior="资料不足时拒答，不编造依据。",
        required_citations=[],
        risk_tags=["no_answer", "refusal"],
        rubric="refusal_v1",
        required_json_fields=["answer", "citations", "needs_human_review"],
        expected_refusal=True,
        source_group="negative_cases",
    ),
]

knowledge_base = {
    "contract_guideline#chunk_1": "liability cap matters for high risk contract clauses"
}

[example.to_dict() for example in examples]

## 2. 运行一个可审计评测

教学版 predictor 用规则函数模拟模型输出，重点观察 runner 如何保存原始输出、解析结果和指标。

In [ ]:
def student_predict(example):
    if example.id == "eval_contract_001":
        return json.dumps(
            {
                "answer": "liability cap matters, should be reviewed.",
                "citations": ["contract_guideline#chunk_1"],
                "needs_human_review": True,
            },
            ensure_ascii=False,
        )
    return json.dumps(
        {
            "answer": "资料不足，无法判断，需要人工复核。",
            "citations": [],
            "needs_human_review": True,
        },
        ensure_ascii=False,
    )

with TemporaryDirectory() as tmpdir:
    result = run_eval(
        examples=examples,
        predict_fn=student_predict,
        output_dir=tmpdir,
        model_id="student-v1",
        knowledge_base=knowledge_base,
        generation_config={"temperature": 0.2, "max_new_tokens": 256},
    )
    print(result.metrics)
    print(Path(result.prediction_path).read_text())
    print(Path(result.report_path).read_text())

## 3. Failure Cases

当模型输出非法 JSON、缺 citation 或拒答错误时，runner 会把样本写入 failure cases。

In [ ]:
def broken_predict(example):
    if example.id == "eval_contract_001":
        return "这个条款风险很高，但我不给 citation。"
    return json.dumps(
        {"answer": "可以直接回答。", "citations": [], "needs_human_review": False},
        ensure_ascii=False,
    )

with TemporaryDirectory() as tmpdir:
    broken = run_eval(
        examples=examples,
        predict_fn=broken_predict,
        output_dir=tmpdir,
        model_id="broken-base",
        knowledge_base=knowledge_base,
    )
    print(broken.metrics)
    print(Path(broken.failure_cases_path).read_text())

## 4. 回归评测

同一 eval set 上比较旧模型和新模型，看指标变化方向，而不是只看新版本平均分。

In [ ]:
old_metrics = {"json_valid": 0.5, "citation_supported": 0.5, "refusal_correct": 0.5}
new_metrics = {"json_valid": 1.0, "citation_supported": 1.0, "refusal_correct": 1.0}
deltas = compare_metric_reports(old_metrics, new_metrics)

with TemporaryDirectory() as tmpdir:
    path = Path(tmpdir) / "regression.md"
    write_regression_report(path, "broken-base", "student-v1", deltas)
    print(path.read_text())